# Define an area and analyse mean values of the spatial distribution of HOSTRADA climate variables

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import Polygon
from pyproj import Transformer
from hostrada4py import hostradaArea as ha
from hostrada4py import hostradaCities as hs
from hostrada4py import hostradaRegions as hr

import os
#os.environ["HOSTRADA_NETCDF_SUBSET_MODE"] = "full"
os.environ["HOSTRADA_NETCDF_SUBSET_MODE"] = "auto"

## Selection of the HOSTRADA value 

In [ ]:
#HOSTRADA_VAR = "tas" # outside air temperature
#HOSTRADA_VAR = "sfcWind" # wind speed
#HOSTRADA_VAR = "sfcWind_direction" # wind direction
HOSTRADA_VAR = "uhi" # Urban Heat Island Intensity
#HOSTRADA_VAR = "rsds" # global radiation
#HOSTRADA_VAR = "clt" # cloud cover
#HOSTRADA_VAR = "hurs" # relative humidity
#HOSTRADA_VAR = "mixr" # Water vapor mixing ratio
#HOSTRADA_VAR = "tdew" # dew point temperature

## Definition of the area and the time period of the HOSTRADA values

In [ ]:
# A list of polygons which include the areas of the 50 largest cities in Germany (and in addition the smaller cities Speyer and Korbach)
polygon_points = hs.berlin_polygon
#polygon_points = hs.hamburg_polygon
#polygon_points = hs.muenchen_polygon
#polygon_points = hs.koeln_polygon
#polygon_points = hs.frankfurt_am_main_polygon
#polygon_points = hs.duesseldorf_polygon
#polygon_points = hs.stuttgart_polygon
#polygon_points = hs.leipzig_polygon
#polygon_points = hs.dortmund_polygon
#polygon_points = hs.bremen_polygon
#polygon_points = hs.essen_polygon
#polygon_points = hs.dresden_polygon
#polygon_points = hs.nuernberg_polygon
#polygon_points = hs.hannover_polygon
#polygon_points = hs.duisburg_polygon
#polygon_points = hs.bochum_polygon
#polygon_points = hs.wuppertal_polygon
#polygon_points = hs.bielefeld_polygon
#polygon_points = hs.bonn_polygon
#polygon_points = hs.mannheim_polygon
#polygon_points = hs.karlsruhe_polygon
#polygon_points = hs.muenster_polygon
#polygon_points = hs.augsburg_polygon
#polygon_points = hs.wiesbaden_polygon
#polygon_points = hs.gelsenkirchen_polygon
#polygon_points = hs.moenchengladbach_polygon
#polygon_points = hs.aachen_polygon
#polygon_points = hs.braunschweig_polygon
#polygon_points = hs.kiel_polygon
#polygon_points = hs.chemnitz_polygon
#polygon_points = hs.magdeburg_polygon
#polygon_points = hs.freiburg_im_breisgau_polygon
#polygon_points = hs.krefeld_polygon
#polygon_points = hs.halle_saale_polygon
#polygon_points = hs.mainz_polygon
#polygon_points = hs.erfurt_polygon
#polygon_points = hs.luebeck_polygon
#polygon_points = hs.oberhausen_polygon
#polygon_points = hs.rostock_polygon
#polygon_points = hs.kassel_polygon
#polygon_points = hs.hagen_polygon
#polygon_points = hs.potsdam_polygon
#polygon_points = hs.saarbruecken_polygon
#polygon_points = hs.hamm_polygon
#polygon_points = hs.oldenburg_oldb_polygon
#polygon_points = hs.ludwigshafen_am_rhein_polygon
#polygon_points = hs.muelheim_an_der_ruhr_polygon
#polygon_points = hs.leverkusen_polygon
#polygon_points = hs.darmstadt_polygon
#polygon_points = hs.korbach_polygon
#polygon_points = hs.speyer_polygon

# A small list of polygons which include the areas of some regions in Germany
#polygon_points = hr.boitzenburgerland_polygon
#polygon_points = hr.uckermark_polygon

start_utc = "2020-08-01T00:00:00"
end_utc = "2020-08-08T23:00:00"

## Download of the HOSTRADA values

In [ ]:
gdf = ha.extract_mean_values_for_polygon(
    var= HOSTRADA_VAR,
    polygon_lonlat=polygon_points,
    start_utc=start_utc,
    end_utc=end_utc,
    selection_mode="within",
    return_geodataframe=True)
print(f"Number of data records: {len(gdf)}")

summary = ha.summarize_values_period(gdf, var = HOSTRADA_VAR)

gdf.to_file("HOSTRADA_poly_" + HOSTRADA_VAR + ".geojson", driver="GeoJSON")
gdf.drop(columns="geometry").to_csv("HOSTRADA_poly_" + HOSTRADA_VAR + ".csv", index=False)
summary.to_csv("HOSTRADA_poly_summary_" + HOSTRADA_VAR + ".csv", index=False)
print("\nResults stored:")
print(" - HOSTRADA_poly_" + HOSTRADA_VAR + ".geojson")
print(" - HOSTRADA_poly_" + HOSTRADA_VAR + ".csv")
print(" - HOSTRADA_poly_summary_" + HOSTRADA_VAR + ".csv")

## Spatial distribution of the HOSTRADA variable over a time period

In [ ]:
if HOSTRADA_VAR == "tas":
    title = "Air temperature in °C"
elif HOSTRADA_VAR == "uhi":
    title = "Urban Heat Island Intensity in °C"
elif HOSTRADA_VAR == "sfcWind":
    title = "Wind speed in m/s"
elif HOSTRADA_VAR == "sfcWind_direction":
    title = "Wind direction in degree"
elif HOSTRADA_VAR == "rsds":
    title = "Global radiation in W/m2"
elif HOSTRADA_VAR == "clt":
    title = "Cloud cover in eighth"
elif HOSTRADA_VAR == "hurs":
    title = "Relative humidity in percent"
elif HOSTRADA_VAR == "tdew":
    title = "Dew point temperature in °C"
elif HOSTRADA_VAR == "mixr":
    title = "Water vapor mixing ratio in g H20/kg dry air"
else:
    title = "unknown"

leaflet_map = ha.make_leaflet_map_timeperiod(
    gdf_or_df=gdf,
    var=HOSTRADA_VAR,
    show_cell_values=True,# Werte im Quadrat anzeige
    decimals=1,# eine Nachkommastelle
    fill_opacity=0.2,# halbtransparent
    value_label_color="black",
    title=title,
    reverse_colormap=False, # klein = blau, groß = rot
    vmin=0.0,
    vmax=3.0,
    save_html='./html/HOSTRADA_' + HOSTRADA_VAR + '.html'
)
leaflet_map